# Supervised Learning: Regression

Regression = wir sagen eine **kontinuierliche Zahl** vorher (z.B. Hauspreise, Temperaturen).

## Inhaltsverzeichnis
1. Der Boston-Housing-Datensatz
2. Lineare Regression
3. Ridge-Regression (L2-Regularisierung)
4. Lasso-Regression (L1-Regularisierung)
5. Decision Tree Regression
6. KNN Regression
7. Bewertungsmetriken: MAE, MSE, RMSE, R²


## 1. Der Boston-Housing-Datensatz

Dieser klassische Datensatz enthält Informationen über Häuser in Boston.  
**Ziel:** Vorhersage des Medianpreises eines Hauses (in 1000er USD)

**Merkmale (Auswahl):**
- `CRIM`: Kriminalitätsrate pro Kopf
- `RM`: Durchschnittliche Zimmeranzahl
- `LSTAT`: Prozentsatz einkommensschwacher Bevölkerung
- `DIS`: Entfernung zu Beschäftigungszentren


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Boston-Datensatz laden
dataset = fetch_openml(name='boston', version=1)
X = dataset.data.astype(float)
y = dataset.target.astype(float)

print(f"Datensatzgröße: {X.shape}")
print(f"Zielwert (Hauspreis) - Statistik:")
print(f"  Minimum:    ${y.min()*1000:.0f}")
print(f"  Maximum:    ${y.max()*1000:.0f}")
print(f"  Mittelwert: ${y.mean()*1000:.0f}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)
print(f"\nTraining: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 2. Lineare Regression

**Idee:** Finde eine Gerade (oder Ebene in mehreren Dimensionen), die am besten durch die Datenpunkte passt.

**Formel:** ŷ = w₀ + w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ

- **w₀**: y-Achsenabschnitt (Intercept)
- **w₁ bis wₙ**: Gewichte (Koeffizienten) — wie stark beeinflusst jedes Merkmal die Vorhersage?

**Optimierung:** Minimiere den mittleren quadratischen Fehler (MSE) zwischen Vorhersagen und echten Werten.

**Wann gut?**
- Wenn ein linearer Zusammenhang zwischen Features und Ziel existiert
- Als einfache und interpretierbare Baseline
- Bei vielen Merkmalen (im Vergleich zu Datenpunkten)


In [ ]:
from sklearn.linear_model import LinearRegression

# Modell erstellen und trainieren
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

print(f"R²-Score (Training): {lin_reg.score(X_train, y_train):.4f}")
print(f"R²-Score (Test):     {lin_reg.score(X_test, y_test):.4f}")
print()

# Koeffizienten anzeigen
koeffizienten = pd.Series(lin_reg.coef_, index=X.columns).sort_values()
print("Koeffizienten (Einfluss auf Hauspreis):")
print(koeffizienten.tail(5).to_string())
print("...")
print(koeffizienten.head(5).to_string())
print()
print("Interpretation: +1 bei LSTAT senkt den vorhergesagten Preis um:", round(koeffizienten['LSTAT'], 2), "Tsd. USD")

In [ ]:
# Vorhersagen vs. echte Werte visualisieren
y_pred = lin_reg.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfekte Vorhersage')
plt.xlabel("Echter Preis (1000 USD)")
plt.ylabel("Vorhergesagter Preis (1000 USD)")
plt.title("Lineare Regression: Vorhersage vs. Realität")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Ridge-Regression (L2-Regularisierung)

**Problem der normalen Linearen Regression:** Bei vielen Merkmalen können die Gewichte (Koeffizienten) sehr groß werden → **Overfitting**

**Lösung:** Ridge "bestraft" große Koeffizienten, indem es sie zur Optimierung hinzuzählt.

**Formel:** Minimiere: MSE + α · Σ(wᵢ²)

- **α (alpha)**: Regularisierungsstärke
  - α = 0: Normale Lineare Regression
  - Sehr großes α: Alle Koeffizienten → 0 (Underfitting)

**Analogie:** Wie ein Sparsamkeitsprinzip — das Modell darf komplexe Erklärungen verwenden, wird aber dafür "bestraft" und bevorzugt einfachere Lösungen.


In [ ]:
from sklearn.linear_model import Ridge

# Verschiedene Alpha-Werte testen
alphas = [0.01, 0.1, 1.0, 10, 100, 1000]

print(f"{'Alpha':>8} | {'Train R²':>10} | {'Test R²':>10}")
print("-" * 35)
for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train, y_train)
    print(f"{a:>8} | {ridge.score(X_train, y_train):>10.4f} | {ridge.score(X_test, y_test):>10.4f}")

In [ ]:
# Wie Regularisierung die Koeffizienten beeinflusst
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lin_koeff = LinearRegression().fit(X_train, y_train).coef_
ridge_koeff = Ridge(alpha=100).fit(X_train, y_train).coef_

axes[0].barh(X.columns, lin_koeff, color='steelblue')
axes[0].set_title("Lineare Regression: Koeffizienten")
axes[0].set_xlabel("Koeffizient")

axes[1].barh(X.columns, ridge_koeff, color='salmon')
axes[1].set_title("Ridge (α=100): Koeffizienten kleiner!")
axes[1].set_xlabel("Koeffizient")

plt.tight_layout()
plt.show()

## 4. Lasso-Regression (L1-Regularisierung)

**Ähnlich wie Ridge**, aber mit einem entscheidenden Unterschied:

**Lasso setzt unwichtige Koeffizienten auf exakt 0!** → automatische **Feature Selection**

**Formel:** Minimiere: MSE + α · Σ|wᵢ|

**Wann Lasso vs. Ridge?**
- **Lasso:** Du glaubst, dass viele Merkmale irrelevant sind und willst ein spärliches Modell
- **Ridge:** Du glaubst, dass alle Merkmale etwas beitragen, nur nicht zu viel


In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=1.0)
lasso.fit(X_train, y_train)

print(f"Train R²: {lasso.score(X_train, y_train):.4f}")
print(f"Test R²:  {lasso.score(X_test, y_test):.4f}")
print()
print("Koeffizienten — Lasso setzt manche auf 0!")
lasso_koeff = pd.Series(lasso.coef_, index=X.columns)
print(lasso_koeff.to_string())
print()
print(f"Anzahl Features mit Koeffizient = 0: {(lasso_koeff == 0).sum()}")
print(f"Anzahl Features die behalten werden: {(lasso_koeff != 0).sum()}")

## 5. Decision Tree Regression

**Idee:** Derselbe Algorithmus wie bei der Klassifikation, aber statt einer Klasse gibt er den **Mittelwert** der Datenpunkte im jeweiligen Blatt zurück.

**Vorteil:** Kann nicht-lineare Zusammenhänge modellieren!
**Nachteil:** Starkes Overfitting ohne `max_depth`


In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Ohne Limit
tree = DecisionTreeRegressor(random_state=1)
tree.fit(X_train, y_train)
print(f"Kein max_depth:    Train R²: {tree.score(X_train, y_train):.4f} | Test R²: {tree.score(X_test, y_test):.4f}")

# Mit Limit
for tiefe in [3, 5, 7]:
    tree = DecisionTreeRegressor(max_depth=tiefe, random_state=1)
    tree.fit(X_train, y_train)
    print(f"max_depth={tiefe}:       Train R²: {tree.score(X_train, y_train):.4f} | Test R²: {tree.score(X_test, y_test):.4f}")

## 6. KNN Regression

**Idee:** Nimm die k nächsten Nachbarn und berechne deren **Mittelwert** als Vorhersage.

**Wichtig:** KNN braucht normalierte Daten! Sonst dominieren Merkmale mit großen Wertebereichen.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MinMaxScaler

# Normalisierung ist bei KNN wichtig!
scaler = MinMaxScaler()
scaler.fit(X_train)  # NUR auf Trainingsdaten fitten!
X_train_norm = scaler.transform(X_train)
X_test_norm = scaler.transform(X_test)

knn_reg = KNeighborsRegressor(n_neighbors=5, weights='distance')
knn_reg.fit(X_train_norm, y_train)

print(f"KNN Regression:")
print(f"  Train R²: {knn_reg.score(X_train_norm, y_train):.4f}")
print(f"  Test R²:  {knn_reg.score(X_test_norm, y_test):.4f}")

## 7. Bewertungsmetriken für Regression

### MAE — Mean Absolute Error (Mittlerer absoluter Fehler)
- **Formel:** (1/n) · Σ|yᵢ - ŷᵢ|
- **Bedeutung:** Durchschnittlicher Fehler in der Einheit der Zielvariable
- **Vorteil:** Leicht interpretierbar! "Im Durchschnitt liege ich X Tsd. USD daneben"
- **Nachteil:** Bestraft große Fehler nicht extra

### MSE — Mean Squared Error (Mittlerer quadratischer Fehler)
- **Formel:** (1/n) · Σ(yᵢ - ŷᵢ)²
- **Bedeutung:** Größere Fehler werden quadratisch bestraft
- **Vorteil:** Empfindlich für Ausreißer (gut wenn man große Fehler vermeiden will)
- **Nachteil:** Schwer zu interpretieren (falsche Einheit: USD²!)

### RMSE — Root Mean Squared Error
- **Formel:** √MSE
- **Bedeutung:** Wie MSE, aber wieder in der richtigen Einheit
- **Vorteil:** Kombination aus interpretierbar + empfindlich für Ausreißer

### R² — Bestimmtheitsmaß (Coefficient of Determination)
- **Formel:** 1 - MSE(Modell) / MSE(Basislinie)
- **Bereich:** -∞ bis 1; 1 = perfekt, 0 = kein besser als Mittelwert, <0 = schlechter als Mittelwert
- **Bedeutung:** Wie viel Prozent der Varianz erklärt das Modell?


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Modell auswählen: Lineare Regression
lin_reg = LinearRegression().fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("Bewertungsmetriken — Lineare Regression:")
print(f"  MAE:  {mae:.2f} Tsd. USD  ← durchschnittlicher Fehler")
print(f"  MSE:  {mse:.2f} (in USD²) ← schwer zu interpretieren")
print(f"  RMSE: {rmse:.2f} Tsd. USD ← besser interpretierbar, bestraft Ausreißer")
print(f"  R²:   {r2:.4f}            ← Modell erklärt {r2:.1%} der Varianz")

In [ ]:
# Alle Modelle vergleichen
from sklearn.metrics import mean_absolute_error, r2_score

modelle = {
    'Lineare Regression': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1),
    'Lasso (α=1)': Lasso(alpha=1),
    'Decision Tree (depth=5)': DecisionTreeRegressor(max_depth=5, random_state=1),
}

print(f"{'Modell':<30} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 60)
for name, modell in modelle.items():
    modell.fit(X_train, y_train)
    y_pred = modell.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    print(f"{name:<30} {mae:>8.2f} {rmse:>8.2f} {r2:>8.4f}")